# Extract Products from CSV

Load the giftcart_cleaned.csv file and extract body_html content into a products array.

In [4]:
# Imports
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
from typing import List
import os

import pandas as pd
load_dotenv()

from huggingface_hub import login
login(token=os.getenv("HF_ACCESS_TOKEN"))

In [3]:
# Read CSV file
df = pd.read_csv('giftcart_cleaned.csv')

# Display basic info
print(f"Total products: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows of body_html:")
df['body_html'].head()

Total products: 2504

Columns: ['Id', 'title', 'handle', 'body_html', 'tags']

First few rows of body_html:


0    We use an incredible blend of milk and dark ch...
1    The "K" Initial Stud Earring for Women is a st...
2    The "S" Initial Stud Earring for Women is a st...
3    Make your mom feel like the true champion she ...
4    A perfect gift for Mom to make them feel speci...
Name: body_html, dtype: str

In [8]:
# Create products array from body_html column, filtering out NaN values
products = df['body_html'].dropna().tolist()

print(f"Created products array with {len(products)} items (filtered out empty values)")
print(f"Original row count: {len(df)}")
print(f"\nFirst product preview:")
print(products[0][:500] + "..." if len(products[0]) > 500 else products[0])

Created products array with 2491 items (filtered out empty values)
Original row count: 2504

First product preview:
We use an incredible blend of milk and dark chocolates. The subsequent chocolate has a rich surface and fully delectable in taste without being excessively sweet. The Chocolates are pure vegetarian Our product are made with utmost care and delicacy with keeping high standard of maintaining the hygiene. Each chocolates weigh about 10-15gms For printed/photo chocolate:The Printing will be done on the chocolate with edible ink which is safe for consumption. Ingredients that are used for printing on...


In [9]:
# Access individual products
print(f"Total products: {len(products)}")
print(f"\nExample - Product at index 90:")
print(products[90])

Total products: 2491

Example - Product at index 90:
Get in touch with our team for more information atcx@giftcart.comor whatsapp at +91 9910644899 Price quoted is exclusive of applicable taxes and shipping charges. Minimum order quantity: 100 pieces Payment Terms: 100% Advance Delivery Timeline: Dispatched within 7 to 8 days from the date of final approval. Dimensions: Volume:Material: Product Weight: gmPackage Weight: gmNumber of Items: 3Marketed By:Giftcart Ecommerce Pvt LtdCustomer Care:+91-9910644899Email: cx@giftcart.comCountry of Origin:India


In [11]:
class MyEmbeddings:
    def __init__(self, model):
        self.model = SentenceTransformer(model, trust_remote_code=True)
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.model.encode(t).tolist() for t in texts]
            
    def embed_query(self, query: str) -> List[float]:
        # Encode single query - returns a single embedding vector
        return self.model.encode(query).tolist()
    
embeddings=MyEmbeddings('google/embeddinggemma-300m')

# Create Document objects with metadata
documents = []
for idx, product in enumerate(products):
    doc = Document(
        page_content=product,
        metadata={"product_id": idx}
    )
    documents.append(doc)

try:
    chromadb = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory="./chroma_db",  # Persist to disk
        collection_name="product_catalog"
    )
    print(f"Created ChromaDB vector store with {len(documents)} products!")
except Exception as e:
    print(f"Error setting up ChromaDB: {str(e)}")


Created ChromaDB vector store with 2491 products!


# Load Existing DB

In [5]:
class MyEmbeddings:
    def __init__(self, model):
        self.model = SentenceTransformer(model, trust_remote_code=True)
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.model.encode(t).tolist() for t in texts]
            
    def embed_query(self, query: str) -> List[float]:
        # Encode single query - returns a single embedding vector
        return self.model.encode(query).tolist()
    
embeddings=MyEmbeddings('google/embeddinggemma-300m')
chromadb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings, collection_name="product_catalog")

In [6]:
# Create retriever
retriever = chromadb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # K is the amount of products to return
)

def retriever_tool(query: str) -> str:
    """
    This tool searches and returns relevant products based on the query.
    """
    docs = retriever.invoke(query)

    if not docs:
        return "I found no relevant products."
    
    results = []
    for i, doc in enumerate(docs):
        product_id = doc.metadata.get('product_id', 'Unknown')
        results.append(f"Product {i+1} (ID: {product_id}):\n{doc.page_content}")
    
    return "\n\n".join(results)

In [8]:
retriever_tool(" Initial Stud Earring for Women")

'Product 1 (ID: 1):\nThe "K" Initial Stud Earring for Women is a stylish and personalized accessory that adds a touch of elegance to any look. Crafted with precision, this stud earring features a beautifully designed letter "K," making it an ideal choice for those who want to showcase their initial or wear a meaningful symbol. Made from high-quality materials such as sterling silver, gold-plated brass, or even diamond-encrusted designs, it ensures durability and long-lasting shine. With its minimalist yet sophisticated appeal, this earring complements both casual and formal outfits. Whether you\'re heading to work, a social gathering, or a special occasion, the "K" Initial Stud Earring is a versatile addition to your jewelry collection. It also makes for a thoughtful and personalized gift for birthdays, anniversaries, or any memorable celebration. Designed for comfort, the earring comes with a secure back closure to ensure a snug fit throughout the day. Lightweight and easy to wear, it